# Phase 7: CNNs — Convolutional Neural Networks

## What you'll learn
- How convolutions work
- Build a CNN from scratch
- Train on CIFAR-10
- Transfer learning with pretrained models
- Fine-tuning vs feature extraction
- Visualize filters and feature maps

---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 7.1 How Convolutions Work

A convolution slides a small **kernel** (filter) across the input to detect patterns.

```
Input (1, 6, 6) → Conv2d(1, 1, kernel=3) → Output (1, 4, 4)
```

Key parameters:
- `in_channels` — input depth (1 for grayscale, 3 for RGB)
- `out_channels` — number of filters (features to detect)
- `kernel_size` — filter size (3×3 is most common)
- `stride` — step size (default 1)
- `padding` — zero-pad input to control output size

In [ ]:
# Output size formula: (W - K + 2P) / S + 1
# W=input, K=kernel, P=padding, S=stride

x = torch.randn(1, 1, 6, 6)  # (batch, channels, H, W)

conv_no_pad = nn.Conv2d(1, 1, kernel_size=3)           # (6-3+0)/1+1 = 4
conv_same = nn.Conv2d(1, 1, kernel_size=3, padding=1)  # (6-3+2)/1+1 = 6
conv_stride = nn.Conv2d(1, 1, kernel_size=3, stride=2, padding=1)  # (6-3+2)/2+1 = 3

print(f"No padding:  {x.shape} → {conv_no_pad(x).shape}")
print(f"Same padding: {x.shape} → {conv_same(x).shape}")
print(f"Stride 2:    {x.shape} → {conv_stride(x).shape}")

## 7.2 Pooling Layers

Pooling reduces spatial dimensions (downsampling) while keeping important features.

In [ ]:
x = torch.randn(1, 16, 32, 32)

maxpool = nn.MaxPool2d(kernel_size=2, stride=2)   # halves spatial dims
avgpool = nn.AvgPool2d(kernel_size=2, stride=2)
adaptive = nn.AdaptiveAvgPool2d((1, 1))            # output always 1x1

print(f"MaxPool2d:     {x.shape} → {maxpool(x).shape}")
print(f"AvgPool2d:     {x.shape} → {avgpool(x).shape}")
print(f"AdaptiveAvg:   {x.shape} → {adaptive(x).shape}")

## 7.3 Build a CNN from Scratch

Classic pattern: **Conv → ReLU → Pool** (repeat) → **Flatten → FC**

In [ ]:
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 3x32x32 → 32x16x16
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 2: 32x16x16 → 64x8x8
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Block 3: 64x8x8 → 128x4x4
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = CNN().to(device)
dummy = torch.randn(2, 3, 32, 32).to(device)
print(f"Output: {model(dummy).shape}")  # (2, 10)

total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")

## 7.4 Train on CIFAR-10

In [ ]:
# Data with augmentation
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

train_set = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=train_transform)
test_set = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=0)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=0)

classes = train_set.classes
print(f"Classes: {classes}")

In [ ]:
model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

for epoch in range(15):
    model.train()
    running_loss = 0
    correct, total = 0, 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += inputs.size(0)

    scheduler.step()

    # Evaluate
    model.eval()
    test_correct, test_total = 0, 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            test_correct += (outputs.argmax(1) == labels).sum().item()
            test_total += inputs.size(0)

    print(f"Epoch {epoch+1:2d}: train_acc={correct/total:.4f} test_acc={test_correct/test_total:.4f}")

## 7.5 Transfer Learning with Pretrained Models

Use a model pretrained on ImageNet and adapt it to your task. Two strategies:

1. **Feature extraction** — freeze all layers, replace final classifier
2. **Fine-tuning** — unfreeze some/all layers, train with small LR

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

# Load pretrained ResNet18
model = resnet18(weights=ResNet18_Weights.DEFAULT)

# --- Feature Extraction: freeze everything ---
for param in model.parameters():
    param.requires_grad = False

# Replace final FC layer (unfrozen by default)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)  # 10 classes for CIFAR-10

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({trainable/total:.1%})")

In [ ]:
# --- Fine-tuning: unfreeze later layers ---
model = resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 10)

# Freeze early layers, unfreeze layer3, layer4, fc
for name, param in model.named_parameters():
    if 'layer3' in name or 'layer4' in name or 'fc' in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

model = model.to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Fine-tuning trainable: {trainable:,}")

## 7.6 Visualize Filters & Feature Maps

In [ ]:
model = CNN().to(device)

# Get first conv layer filters
filters = model.features[0].weight.data.cpu()
print(f"First conv filters: {filters.shape}")  # (32, 3, 3, 3)

# Visualize first 8 filters
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    if i < filters.shape[0]:
        # Normalize filter for display
        f = filters[i].permute(1, 2, 0)  # CHW → HWC
        f = (f - f.min()) / (f.max() - f.min())
        ax.imshow(f)
    ax.axis('off')
    ax.set_title(f'Filter {i}')
plt.suptitle('First Conv Layer Filters')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize feature maps
model.eval()
sample = torch.randn(1, 3, 32, 32).to(device)

# Hook to capture intermediate output
activation = {}
def hook_fn(name):
    def hook(module, input, output):
        activation[name] = output.detach().cpu()
    return hook

model.features[0].register_forward_hook(hook_fn('conv1'))
_ = model(sample)

# Plot first 8 feature maps
feat = activation['conv1'][0]  # remove batch dim
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(feat[i], cmap='viridis')
    ax.axis('off')
    ax.set_title(f'Map {i}')
plt.suptitle('Feature Maps after Conv1')
plt.tight_layout()
plt.show()

## ✅ Phase 7 Summary

You now know how to:
- Build CNNs with Conv → BN → ReLU → Pool blocks
- Calculate output sizes with the formula
- Train on real image datasets (CIFAR-10)
- Use pretrained models for transfer learning
- Choose between feature extraction and fine-tuning
- Visualize what the network learns

**Next → Phase 8: RNNs & Sequence Models**